# H&M Personalized Fashion Recommendations — EDA

## Competition Overview
- **Task**: Predict 12 articles each customer will buy in the next 7 days
- **Metric**: MAP@12
- **Data**: 2 years of transactions (2018-09 to 2020-09), 105K articles, 1.37M customers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA = Path('../data_raw')
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Articles (105K)

In [ ]:
articles = pd.read_csv(DATA / 'articles.csv')
print(f'Shape: {articles.shape}')
print(f'Columns: {list(articles.columns)}')
articles.head(2)

In [ ]:
# Article categorical distributions
cat_cols = ['product_group_name', 'index_group_name', 'section_name', 'garment_group_name', 'perceived_colour_master_name']
for col in cat_cols:
    print(f'\n{col}: {articles[col].nunique()} unique')
    print(articles[col].value_counts().head(10))

## 2. Customers (1.37M)

In [ ]:
customers = pd.read_csv(DATA / 'customers.csv')
print(f'Shape: {customers.shape}')
print(f'Columns: {list(customers.columns)}')
customers.head(2)

In [ ]:
# Customer demographics
print('Age distribution:')
print(customers['age'].describe())
print(f"\nAge NaN: {customers['age'].isna().sum()} ({customers['age'].isna().mean()*100:.1f}%)")
print(f"\nclub_member_status:\n{customers['club_member_status'].value_counts()}")
print(f"\nfashion_news_frequency:\n{customers['fashion_news_frequency'].value_counts()}")
print(f"\nFN (fashion news): {customers['FN'].notna().mean()*100:.1f}% have it")
print(f"Active: {customers['Active'].notna().mean()*100:.1f}% active")

In [ ]:
customers['age'].dropna().hist(bins=50)
plt.title('Customer Age Distribution')
plt.xlabel('Age')
plt.show()

## 3. Transactions (31.8M) — Sample First

In [ ]:
# Read first 1M rows to understand structure
txn = pd.read_csv(DATA / 'transactions_train.csv', nrows=1_000_000, parse_dates=['t_dat'])
print(f'Shape (sample): {txn.shape}')
print(f'Columns: {list(txn.columns)}')
print(f'\nDate range: {txn["t_dat"].min()} to {txn["t_dat"].max()}')
txn.head()

In [ ]:
# Transaction stats
print(f'Unique customers: {txn["customer_id"].nunique():,}')
print(f'Unique articles: {txn["article_id"].nunique():,}')
print(f'\nSales channel:\n{txn["sales_channel_id"].value_counts()}')
print(f'\nPrice stats:\n{txn["price"].describe()}')

In [ ]:
# Weekly transaction volume
txn['week'] = txn['t_dat'].dt.isocalendar().week.astype(int)
txn['year_week'] = txn['t_dat'].dt.strftime('%Y-W%W')
weekly = txn.groupby('year_week').size()
weekly.plot(kind='bar', figsize=(16, 4))
plt.title('Weekly Transaction Volume (first 1M rows)')
plt.xticks([])
plt.show()

## 4. Full Data Stats (using bash)

In [ ]:
import subprocess
# Count lines in full transactions file
result = subprocess.run(['wc', '-l', str(DATA / 'transactions_train.csv')], capture_output=True, text=True)
txn_lines = int(result.stdout.split()[0])
print(f'Transactions rows: {txn_lines - 1:,} (excluding header)')

result = subprocess.run(['wc', '-l', str(DATA / 'sample_submission.csv')], capture_output=True, text=True)
sub_lines = int(result.stdout.split()[0])
print(f'Sample submission rows: {sub_lines - 1:,}')

## 5. Target Period Analysis

In [ ]:
# Full date range
txn_full_dates = pd.read_csv(DATA / 'transactions_train.csv', usecols=['t_dat'], parse_dates=['t_dat'])
print(f'Full date range: {txn_full_dates["t_dat"].min()} to {txn_full_dates["t_dat"].max()}')

# Last 7 days = test period
max_date = txn_full_dates['t_dat'].max()
test_start = max_date - pd.Timedelta(days=6)
print(f'\nTest period (last 7 days): {test_start.date()} to {max_date.date()}')
test_period = txn_full_dates[txn_full_dates['t_dat'] >= test_start]
print(f'Transactions in last 7 days: {len(test_period):,}')
print(f'Unique customers in last 7 days: {test_period.nunique()["t_dat"] if "customer_id" not in test_period.columns else "N/A (need to load customer_id)"}')
del txn_full_dates

In [ ]:
# Load with customer_id for last week analysis
txn_tail = pd.read_csv(DATA / 'transactions_train.csv', parse_dates=['t_dat'])
max_date = txn_tail['t_dat'].max()
test_start = max_date - pd.Timedelta(days=6)

last_week = txn_tail[txn_tail['t_dat'] >= test_start]
prev_week = txn_tail[(txn_tail['t_dat'] >= test_start - pd.Timedelta(weeks=1)) & (txn_tail['t_dat'] < test_start)]

print(f'Last week: {len(last_week):,} txns, {last_week["customer_id"].nunique():,} customers, {last_week["article_id"].nunique():,} articles')
print(f'Prev week: {len(prev_week):,} txns, {prev_week["customer_id"].nunique():,} customers')

# Repurchase rate: customers who bought in prev week AND last week
prev_customers = set(prev_week['customer_id'].unique())
last_customers = set(last_week['customer_id'].unique())
repeat = prev_customers & last_customers
print(f'\nCustomers in both weeks: {len(repeat):,} ({len(repeat)/len(last_customers)*100:.1f}% of last week customers)')

## 6. Submission Format

In [ ]:
sub = pd.read_csv(DATA / 'sample_submission.csv', nrows=5)
print(f'Submission columns: {list(sub.columns)}')
print(sub)

In [ ]:
# Parse the prediction format
pred_str = sub['prediction'].iloc[0]
preds = pred_str.split()
print(f'Predictions per customer: {len(preds)}')
print(f'Sample predictions: {preds[:5]}...')

## 7. Key Insights for Baseline

In [ ]:
print('=== Key Insights ===')
print('1. Prediction format: 12 space-separated article_ids per customer')
print('2. Need to predict purchases in 7 days after training period ends')
print('3. Two-stage approach: candidate generation → ranking')
print('4. Cold start: ~50% test customers may have no recent transactions')
print('5. Fashion is trendy: recent popularity matters more than historical')
print('6. Repurchase is strong signal (10.8% buy same item within a week)')